# AI Risk Manager — Model Training

This notebook trains both stages of the fraud detection pipeline:

- **Stage 1**: XGBoost-based binary risk scorer (legitimate vs fraud)
- **Stage 2**: Random Forest multi-class fraud type classifier

It also evaluates model performance, generates visualizations, and performs cost analysis using the project's INR-denominated cost matrix.

In [ ]:
# AI Risk Manager - Model Training
# Stage 1: Risk Scorer + Stage 2: Fraud Classifier

import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from data.generate import generate_transactions
from models.feature_engine import FeatureEngine, FEATURES
from models.cost_matrix import COST_MATRIX, calculate_cost, optimize_threshold
from models.stage1_risk_scorer import Stage1RiskScorer
from models.stage2_fraud_classifier import Stage2FraudClassifier

## Load and Prepare Data

Load the synthetic transaction dataset, inspect class balance, and split into train/test sets with stratification to preserve the fraud rate.

In [ ]:
# Load data
df = pd.read_csv("../data/transactions.csv")
print(f"Dataset shape: {df.shape}")
print(f"Chargeback rate: {df['chargeback_label'].mean():.1%}")

# Split data
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['chargeback_label'])
print(f"\nTrain set: {len(train_df)} transactions")
print(f"Test set: {len(test_df)} transactions")

## Feature Engineering

Fit the `FeatureEngine` on training data and transform both splits into 20 model-ready features covering velocity, device trust, geographic anomalies, account history, temporal patterns, and amount distributions.

In [ ]:
# Initialize and fit feature engine
feature_engine = FeatureEngine()
feature_engine.fit(train_df)

# Transform both sets
train_features = feature_engine.transform(train_df)
test_features = feature_engine.transform(test_df)

print(f"Feature count: {len(FEATURES)}")
print(f"Features: {FEATURES}")
print(f"\nTrain features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")

## Stage 1 — Risk Scorer Training

Train an XGBoost binary classifier with cost-optimized threshold selection. The model uses `scale_pos_weight` to handle class imbalance and evaluates using the project's INR cost matrix.

In [ ]:
# Train Stage 1
stage1 = Stage1RiskScorer(threshold_mode="cost_optimized")
metrics = stage1.train(train_df, feature_engine)

print("\n=== Stage 1 Training Metrics ===")
for key, val in metrics.items():
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

## Stage 1 — Evaluation

Evaluate the trained Stage 1 model on the held-out test set using precision, recall, F1, AUC-ROC, and AUC-PR metrics.

In [ ]:
# Evaluate on test set
test_features_df = feature_engine.transform(test_df)
X_test = test_features_df[FEATURES].values
y_test = test_features_df["chargeback_label"].astype(int).values

# Get predictions
y_proba = stage1.predict_proba(test_features_df)
y_pred = stage1.predict(test_features_df)

# Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

print("=== Stage 1 Test Metrics ===")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"AUC-PR: {average_precision_score(y_test, y_proba):.4f}")

## Confusion Matrix

Visualize the confusion matrix to understand the distribution of true/false positives and negatives.

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix - Stage 1 (Risk Scorer)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../evaluation/reports/stage1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## ROC and Precision-Recall Curves

Plot ROC and PR curves to assess the model's discriminative ability across all thresholds.

In [ ]:
# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curve - Stage 1')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR Curve
from sklearn.metrics import precision_recall_curve
precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
axes[1].plot(recall, precision, label=f'PR (AUC = {pr_auc:.3f})')
axes[1].set_title('Precision-Recall Curve - Stage 1')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../evaluation/reports/stage1_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Importance

Display the top 10 most important features as ranked by the XGBoost model.

In [ ]:
# Feature importance
importance_df = stage1.get_feature_importance()
print("=== Top 10 Feature Importances ===")
print(importance_df.head(10).to_string(index=False))

plt.figure(figsize=(10, 6))
top_features = importance_df.head(10)
plt.barh(top_features['feature'], top_features['importance'])
plt.title('Top 10 Feature Importances - Stage 1')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../evaluation/reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Stage 2 — Fraud Classifier Training

Train a Random Forest multi-class classifier to categorize detected fraud into: `genuine`, `friendly_fraud`, `account_takeover`, or `technical_failure`. Only fraud-flagged transactions are classified.

In [ ]:
# Train Stage 2
stage2 = Stage2FraudClassifier()
stage2_metrics = stage2.train(train_df, feature_engine)

print("\n=== Stage 2 Classification Report ===")
print(stage2_metrics.get('classification_report', 'N/A'))

## Cost Analysis

Analyze the financial impact of the model's predictions using the INR cost matrix. The cost curve shows how total cost, savings, and net benefit vary across different classification thresholds.

In [ ]:
# Cost analysis
amounts = test_df["amount"].values.astype(np.float64)
cost_result = calculate_cost(y_test, y_pred, amounts)

print("=== Cost Analysis ===")
print(f"Total FN Cost: INR {cost_result['total_fn_cost']:,.0f}")
print(f"Total FP Cost: INR {cost_result['total_fp_cost']:,.0f}")
print(f"Total TP Cost: INR {cost_result['total_tp_cost']:,.0f}")
print(f"Total Cost: INR {cost_result['total_cost']:,.0f}")
print(f"Total Savings: INR {cost_result['total_savings']:,.0f}")
print(f"Net Benefit: INR {cost_result['net_benefit']:,.0f}")

# Cost curve
from models.cost_matrix import CostAnalyzer
analyzer = CostAnalyzer()
cost_curve = analyzer.cost_curve(y_test, y_proba, amounts)

plt.figure(figsize=(10, 6))
plt.plot(cost_curve['threshold'], cost_curve['total_cost'], label='Total Cost')
plt.plot(cost_curve['threshold'], cost_curve['total_savings'], label='Total Savings')
plt.plot(cost_curve['threshold'], cost_curve['net_benefit'], label='Net Benefit', linewidth=2)
plt.axvline(x=stage1.threshold, color='r', linestyle='--', label=f'Optimal Threshold ({stage1.threshold:.2f})')
plt.title('Cost Curve Across Thresholds')
plt.xlabel('Threshold')
plt.ylabel('Amount (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../evaluation/reports/cost_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Training complete. Both models are saved to `models/artifacts/` for use by the API and evaluation pipelines.

In [ ]:
# Training Summary
print("=== Model Training Summary ===")
print(f"\nStage 1 (Risk Scorer):")
print(f"  Model: XGBoost (200 estimators, depth 8)")
print(f"  Threshold: {stage1.threshold:.2f} ({stage1.threshold_mode})")
print(f"  Features: {len(FEATURES)}")
print(f"\nStage 2 (Fraud Classifier):")
print(f"  Model: Random Forest (150 estimators, depth 12)")
print(f"  Classes: 4 (genuine, friendly_fraud, account_takeover, technical_failure)")
print(f"\nModels saved to: models/artifacts/")